# MITSUI Commodity Prediction — Colab training and submission

End-to-end training, validation and sequential inference for all 424 targets.
The pipeline combines target-pair causal features, LightGBM, Random Forest,
XGBoost, time-series OOF stacking and the Kaggle inference server.

## 1. Clone and install

In [ ]:
REPO_URL = "https://github.com/mingzhuoFUN/mitsui-commodity-prediction.git"
# The complete pipeline currently lives on this remote branch. Change to
# "main" after the feature branch is merged.
REPO_REF = "agent/full-competition-pipeline"
!git clone --depth 1 --branch {REPO_REF} {REPO_URL}
%cd mitsui-commodity-prediction
!pip -q install -r requirements.txt
!pip -q install -e .

import sys
from importlib.metadata import version
import mitsui
print("Python:", sys.version)
print("mitsui:", mitsui.__file__)
for package in ("pandas", "scikit-learn", "lightgbm", "xgboost"):
    print(package, version(package))

## 2. Competition data

Recommended authentication: add the current `KGAT_...` token to Colab Secrets
with the name `KAGGLE_API_TOKEN` and enable notebook access. A legacy
`kaggle.json` upload remains available as a fallback. Never place either
credential in GitHub or a notebook cell.

In [ ]:
from pathlib import Path
import os
DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not (DATA_DIR / "train.csv").exists():
    try:
        from google.colab import userdata
        api_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        api_token = None

    if api_token:
        os.environ["KAGGLE_API_TOKEN"] = api_token
    else:
        from google.colab import files
        print("No KAGGLE_API_TOKEN secret found; upload legacy kaggle.json.")
        uploaded = files.upload()
        token_file = next(iter(uploaded))
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        (kaggle_dir / "kaggle.json").write_bytes(uploaded[token_file])
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
    !kaggle competitions download -c mitsui-commodity-prediction-challenge -p data/raw
    !unzip -q -o data/raw/mitsui-commodity-prediction-challenge.zip -d data/raw

!python scripts/inspect_data.py --data-dir data/raw

## 3. Verify official X/Y alignment and target horizons

In [ ]:
import pandas as pd
train = pd.read_csv(DATA_DIR / "train.csv")
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
pairs = pd.read_csv(DATA_DIR / "target_pairs.csv")
assert train["date_id"].equals(labels["date_id"])
display(pairs.groupby("lag").size().rename("targets"))
print("market", train.shape, "labels", labels.shape)

## 4. Leakage checks

Tests include a future-mutation test: changing future market rows must not
change features already computed for past rows.

In [ ]:
!pytest -q

## 5. Inspect the actual model code

The notebook calls repository modules so the same tested implementation is
used in Colab, local validation and Kaggle inference.

In [ ]:
from mitsui.features import make_target_features
from mitsui.ensemble import fit_stacked_models, predict_stacked_models
from mitsui.inference import SequentialPredictor

print("Base models: LightGBM, Random Forest, XGBoost")
print("Meta model: Ridge trained from time-series OOF predictions")

## 6. Eight-target smoke run

In [ ]:
!python scripts/train_ensemble.py \
  --data-dir data/raw \
  --output-dir outputs/ensemble_smoke \
  --valid-size 128 \
  --max-targets 8

## 7. Full chronological validation

Each target uses LightGBM, Random Forest and XGBoost. Their time-series OOF
predictions train a Ridge meta-model. The final 252 rows are untouched until
validation.

In [ ]:
!python scripts/train_ensemble.py \
  --data-dir data/raw \
  --output-dir outputs/ensemble_full \
  --valid-size 252

In [ ]:
import json
json.loads(Path("outputs/ensemble_full/metrics.json").read_text())

## 8. Fit submission models on all official training rows

In [ ]:
!python scripts/train_ensemble.py \
  --data-dir data/raw \
  --output-dir outputs/ensemble_submit \
  --fit-full

## 9. Persist artifacts to Google Drive

Colab runtimes are temporary. Mount Drive and copy the trained model, metrics
and validation predictions before starting the gateway. The copy is verified
by file existence and size.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
ARTIFACT_DIR = Path("/content/drive/MyDrive/mitsui-artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    Path("outputs/ensemble_submit/stacked_models.pkl"): "stacked_models.pkl",
    Path("outputs/ensemble_submit/metrics.json"): "submission_metrics.json",
    Path("outputs/ensemble_full/metrics.json"): "validation_metrics.json",
    Path("outputs/ensemble_full/validation_predictions.csv"):
        "validation_predictions.csv",
}
for source, destination_name in artifacts.items():
    if source.exists():
        destination = ARTIFACT_DIR / destination_name
        shutil.copy2(source, destination)
        assert destination.exists() and destination.stat().st_size > 0
        print(destination, destination.stat().st_size, "bytes")

## 10. Run the complete local Kaggle gateway

In [ ]:
!python scripts/run_local_gateway.py \
  --data-dir data/raw \
  --model-path outputs/ensemble_submit/stacked_models.pkl

## 11. Competition rerun entrypoint

For a Kaggle submission, keep the trained model artifact in a Kaggle Dataset
attached to the notebook, initialize `SequentialPredictor`, then serve it:

In [ ]:
import os, sys
import pandas as pd
from mitsui.ensemble import load_stacked_models
from mitsui.inference import SequentialPredictor

sys.path.append(str(DATA_DIR.resolve()))
from kaggle_evaluation.mitsui_inference_server import MitsuiInferenceServer

predictor = SequentialPredictor(
    load_stacked_models("outputs/ensemble_submit/stacked_models.pkl"),
    pd.read_csv(DATA_DIR / "train.csv"),
    pd.read_csv(DATA_DIR / "train_labels.csv"),
)

def predict(test, label_lags_1, label_lags_2, label_lags_3, label_lags_4):
    return predictor.predict(
        test, label_lags_1, label_lags_2, label_lags_3, label_lags_4
    )

inference_server = MitsuiInferenceServer(predict)
if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    print("Use the previous cell for local gateway validation.")

## Artifacts

`outputs/ensemble_submit/stacked_models.pkl` contains all 424 target bundles.
Store it in Google Drive or a private Kaggle Dataset; it is intentionally not
committed to GitHub.